### Basic Setups

In [1]:
import os
import cv2
import matplotlib.pyplot as plt
import random
from ultralytics import YOLO
import shutil
from pathlib import Path
import yaml

### Data Preprocessing

In [ ]:
RawDataset = Path("Detection Dataset 3 Class/YOLO TXT")      
OrganizedDataset = Path("Detection Dataset 3 Class/YOLO TXT/Master Dataset")  

SplitMapping = {
    "test": ["Camera di tengah Satu Arah Banyak"], 
    "val": ["Jalan Rame"],  
    "train": ["Highway CCTV", "CCTV Lalu Lintas Surabaya", "VEHICLE", "Multi Tracking"]
}

def DirectorySetup(BasePath):
    Splits = ["train", "val", "test"]
    
    if not BasePath.exists():
        BasePath.mkdir(parents=True)
        print(f"Created Master Directory at: {BasePath}")

    for split in Splits:
        split_dir = BasePath / split
        if split_dir.exists():
            print(f"  Cleaning existing split folder: {split}...")
            shutil.rmtree(split_dir)
        
        (split_dir / "images").mkdir(parents=True, exist_ok=True)
        (split_dir / "labels").mkdir(parents=True, exist_ok=True)

def MergeDataset():
    DirectorySetup(OrganizedDataset)
    
    Classes = []
    
    for SplitName, ListOfFolders in SplitMapping.items():
        print(f"\n--- Processing {SplitName.upper()} split ---")
        
        for FolderName in ListOfFolders:
            LocationPath = RawDataset / FolderName
            
            ImageFolder = LocationPath / "train" / "images"
            LabelFolder = LocationPath / "train" / "labels"
            YamlFile = LocationPath / "data.yaml"
            
            if not ImageFolder.exists() or not LabelFolder.exists():
                print(f"  [ERROR] Could not find images/labels in {FolderName}. Skipping.")
                continue
            
            if not Classes and YamlFile.exists():
                with open(YamlFile, 'r') as f:
                    data = yaml.safe_load(f)
                    Classes = data.get('names', [])
            
            images = []
            for ext in ('*.jpg', '*.jpeg', '*.png'):
                images.extend(ImageFolder.glob(ext))
                
            print(f"  Copying {len(images)} files from {FolderName} to {SplitName}...")
            
            for ImagePath in images:
                LabelPath = LabelFolder / f"{ImagePath.stem}.txt"
                
                UpdatedImageName = f"{FolderName}_{ImagePath.name}"
                UpdatedLabelName = f"{FolderName}_{LabelPath.name}"
                
                OutputImagePath = OrganizedDataset / SplitName / "images" / UpdatedImageName
                OutputLabelPath = OrganizedDataset / SplitName / "labels" / UpdatedLabelName
                
                shutil.copy(ImagePath, OutputImagePath)
                
                if LabelPath.exists(): shutil.copy(LabelPath, OutputLabelPath)
                else: open(OutputLabelPath, 'w').close() 

    MasterYaml = {
        'path': str(OrganizedDataset.resolve()),
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'names': Classes
    }
    
    with open(OrganizedDataset / 'data.yaml', 'w') as f:
        yaml.dump(MasterYaml, f, sort_keys=False)
        
    print("\n✅ Master dataset merged successfully!")
    print(f"Master data.yaml created at {OrganizedDataset / 'data.yaml'}")

if __name__ == "__main__":
    MergeDataset()


--- Processing TEST split ---
  Copying 182 files from Camera di tengah Satu Arah Banyak to test...

--- Processing VAL split ---
  Copying 838 files from Jalan Rame to val...

--- Processing TRAIN split ---
  Copying 1092 files from Highway CCTV to train...
  Copying 966 files from CCTV Lalu Lintas Surabaya to train...
  Copying 395 files from VEHICLE to train...
  Copying 200 files from Multi Tracking to train...

✅ Master dataset merged successfully!
Master data.yaml created at Detection Dataset 3 Class\YOLO TXT\Master Dataset\data.yaml


In [2]:
model = YOLO('yolov8n.pt')

model.info(detailed=True)

layer                                    name                type  gradient  parameters               shape        mu     sigma
    0                     model.0.conv.weight              Conv2d     False         432       [16, 3, 3, 3]  -0.00279     0.152        float32
    1                       model.0.bn.weight         BatchNorm2d     False          16                [16]      2.97      1.86        float32
    1                         model.0.bn.bias         BatchNorm2d     False          16                [16]     0.249      4.17        float32
    2                             model.0.act                SiLU     False           0                  []         -         -              -
    3                     model.1.conv.weight              Conv2d     False        4608      [32, 16, 3, 3]  -0.00012     0.063        float32
    4                       model.1.bn.weight         BatchNorm2d     False          32                [32]      5.02      1.12        float32
    4         

(129, 3157200, 0, 8.8575488)

In [ ]:
from ultralytics import YOLO

DataPath = 'Detection Dataset 3 Class/YOLO TXT/Master Dataset/data.yaml'

# ==========================================
# STAGE 1 : WARMUP 
# ==========================================
model = YOLO('yolov8n.pt') 

model.train(
    data         =    DataPath, 
    epochs       =    15,          
    freeze       =    22, 
    lr0          =    0.01,
    project      =    'Detector Result',
    name         =    'TransferLearningDetection',
    device       =    0,
    imgsz        =    800,         
    amp          =    True,
    batch        =    8,  
    workers      =    2,
    cache        =    False,      
    patience     =    10,       
    exist_ok     =    True,
    
    # --- AUGMENTATION ---
    cls          =     4.0,           
    box          =     7.5, 
    hsv_v        =     0.4,
    scale        =     0.5,
    fliplr       =     0.5,
    mosaic       =     1.0,
    overlap_mask =     True  
)

New https://pypi.org/project/ultralytics/8.4.33 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.222  Python-3.10.18 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=4.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Detection Dataset 3 Class/YOLO TXT/Master Dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=22, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, n

c:\Users\justi\anaconda3\envs\pytorch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access  (ping: 0.10.0 ms, read: 513.766.0 MB/s, size: 397.5 KB)
val: Scanning C:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\CVision\Detection Dataset 3 Class\YOLO TXT\Master Dataset\val\labels.cache... 838 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 838/838 840.7Kit/s 0.0s
Plotting labels to C:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\CVision\Detector Result\TransferLearningDetection\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001429, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 800 train, 800 val
Using 2 

FileNotFoundError: [Errno 2] No such file or directory: 'Detector Result\\BestTransferLearningDetection\\weights\\best.pt'

In [4]:
# ==========================================
# STAGE 2 : FINE-TUNE
# ==========================================
model = YOLO('Detector Result/TransferLearningDetection/weights/best.pt') 

model.train(
    data         =    DataPath, 
    epochs       =    50,          
    freeze       =    0,             
    lr0          =    0.001,
    cos_lr       =    True,      
    project      =    'Detector Result',
    name         =    'FineTunedTransferLearningDetection',
    device       =    0,
    imgsz        =    800,        
    amp          =    True,
    batch        =    8,
    workers      =    2,
    cache        =    False,      
    patience     =    15,
    exist_ok     =    True,
    
    # --- AUGMENTATION ---
    cls          =    5.0,           
    box          =    10.0,          
    hsv_v        =    0.4,
    scale        =    0.8,  
    mosaic       =    1.0,
    close_mosaic =    20,  
    mixup        =    0.15,         
    copy_paste   =    0.3,            
    fliplr       =    0.5
)

New https://pypi.org/project/ultralytics/8.4.33 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.222  Python-3.10.18 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=10.0, cache=False, cfg=None, classes=None, close_mosaic=20, cls=5.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=Detection Dataset 3 Class/YOLO TXT/Master Dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=Detector Result/TransferLearningDetection/weights/best.pt, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000240C5C88B80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          

In [6]:
ModelTest = YOLO('Detector Result/FineTunedTransferLearningDetection/weights/best.pt')

TestImageFolder = 'Detection Dataset Upgrade/YOLO TXT/Master Dataset/test/images'
TestImages = [os.path.join(TestImageFolder, f) for f in os.listdir(TestImageFolder) if f.endswith(('.jpg', '.png', '.jpeg'))]

SampleImages = random.sample(TestImages, 6)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, ImagePath in enumerate(SampleImages):
    results = ModelTest.predict(source=ImagePath, conf=0.2, save=False)[0]
    
    res_plotted = results.plot()
    
    res_rgb = cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)
    
    axes[i].imshow(res_rgb)
    axes[i].set_title(f"Test Image {i+1}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

ModelTest.predict(source=TestImageFolder, conf=0.2, save=True, project='Detector Result', name='test_visuals')
print(f"All annotated images saved to: Detector Result/test_visuals")


image 1/1 c:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\CVision\Detection Dataset Upgrade\YOLO TXT\Master Dataset\test\images\Camera di tengah Satu Arah Banyak_video9_frame_6_jpg.rf.88cb58a29281bb4d1ec65585eb4ea3e3.jpg: 480x800 17 cars, 14 motorcycles, 22.0ms
Speed: 4.7ms preprocess, 22.0ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 800)

image 1/1 c:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\CVision\Detection Dataset Upgrade\YOLO TXT\Master Dataset\test\images\Camera di tengah Satu Arah Banyak_video4_frame_2_jpg.rf.0f69c42daac14acb91da62386d192608.jpg: 480x800 9 cars, 4 heavy_vehicles, 17 motorcycles, 20.9ms
Speed: 4.3ms preprocess, 20.9ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 800)

image 1/1 c:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\CVision\Detection Dataset Upgrade\YOLO TXT\Master Dataset\test\images\Camera di tengah Satu Arah Banyak_nt_test3_frame_5_jpg.rf.003a1377d50d006d301d5314b

<Figure size 1800x1000 with 6 Axes>

<Figure size 1800x1000 with 6 Axes>


image 1/182 c:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\CVision\Detection Dataset Upgrade\YOLO TXT\Master Dataset\test\images\Camera di tengah Satu Arah Banyak_frame0_jpg.rf.826c392462e23cb09fab9ca7fa47c28c.jpg: 480x800 16 cars, 4 heavy_vehicles, 23 motorcycles, 17.9ms
image 2/182 c:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\CVision\Detection Dataset Upgrade\YOLO TXT\Master Dataset\test\images\Camera di tengah Satu Arah Banyak_frame10_jpg.rf.08e74ad14c0d60e3f98b0e95a05e4348.jpg: 480x800 7 cars, 5 heavy_vehicles, 19 motorcycles, 19.5ms
image 3/182 c:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\CVision\Detection Dataset Upgrade\YOLO TXT\Master Dataset\test\images\Camera di tengah Satu Arah Banyak_frame12_jpg.rf.a23f8c1eed24750cfe034f407cf06f46.jpg: 480x800 15 cars, 1 heavy_vehicle, 32 motorcycles, 18.0ms
image 4/182 c:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\CVision\Detection Dataset Upgrade\YOLO TXT\Master